In [22]:
from google.colab import drive
drive.mount('/content/drive')

import sys, os
sys.path.append("/content/drive/MyDrive/tcc")
from src import config

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!df -h /content | tail -1
!df -h /content/drive | tail -1

overlay         108G   88M  108G   1% /
drive           108G  5.5G  103G   6% /content/drive


In [3]:
ZIP_DRIVE = f"{config.BRUTOS}/synthbuster/synthbuster.zip"

if not os.path.exists(ZIP_DRIVE):
    !wget -c -O /content/synthbuster.zip "https://zenodo.org/records/10066460/files/synthbuster.zip?download=1"
    !cp /content/synthbuster.zip "{ZIP_DRIVE}"
    !rm /content/synthbuster.zip
else:
    print("Zip já está no Drive")

!ls -lh "{ZIP_DRIVE}"

--2026-11-04 14:33:18--  https://zenodo.org/records/10066460/files/synthbuster.zip?download=1
Resolving zenodo.org (zenodo.org)... 188.185.45.63, 188.185.47.192, 188.185.40.132, ...
Connecting to zenodo.org (zenodo.org)|188.185.45.63|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12372557226 (12G) [application/octet-stream]
Saving to: ‘/content/synthbuster.zip’

/content/synthbuste 100%[===================>]  11.52G  1.59MB/s    in 1h 43m  

2026-11-04 16:16:21 (1.91 MB/s) - ‘/content/synthbuster.zip’ saved [12372557226/12372557226]

-rw------- 1 root root 12G Nov  4 16:19 /content/drive/MyDrive/tcc/dados_brutos/synthbuster/synthbuster.zip


In [4]:
if not os.path.exists("/content/synthbuster"):
    !unzip -q "{ZIP_DRIVE}" -d /content/synthbuster
!ls /content/synthbuster

synthbuster


In [5]:
import pandas as pd
from PIL import Image

linhas = []
for raiz, _, arquivos in os.walk("/content/synthbuster"):
    for nome in arquivos:
        if nome.lower().endswith((".png", ".jpg", ".jpeg", ".webp")):
            caminho = os.path.join(raiz, nome)
            with Image.open(caminho) as img:
                largura, altura = img.size
                formato = img.format
            linhas.append({
                "gerador": os.path.relpath(raiz, "/content/synthbuster"),
                "arquivo": nome,
                "largura": largura,
                "altura": altura,
                "formato": formato,
                "tamanho_mb": os.path.getsize(caminho) / 1e6,
            })

inv = pd.DataFrame(linhas)
inv.to_csv(f"{config.BRUTOS}/synthbuster/inventario.csv", index=False)

print(inv.groupby("gerador").agg(
    imagens=("arquivo", "count"),
    largura_min=("largura", "min"),
    altura_min=("altura", "min"),
    formato=("formato", lambda x: ", ".join(sorted(set(x)))),
    total_gb=("tamanho_mb", lambda x: round(x.sum() / 1000, 2)),
))

                                  imagens  largura_min  altura_min formato  \
gerador                                                                      
synthbuster/dalle2                   1000         1024        1024     PNG   
synthbuster/dalle3                   1000         1024        1024     PNG   
synthbuster/firefly                  1000         1792        1536    JPEG   
synthbuster/glide                    1000          256         256     PNG   
synthbuster/midjourney-v5            1000          896         896     PNG   
synthbuster/stable-diffusion-1-3     1000          512         512     PNG   
synthbuster/stable-diffusion-1-4     1000          512         512     PNG   
synthbuster/stable-diffusion-2       1000          704         576     PNG   
synthbuster/stable-diffusion-xl      1000          704         576     PNG   

                                  total_gb  
gerador                                     
synthbuster/dalle2                    3.15  
synthb

In [7]:
import shutil, glob

origem = glob.glob("/content/synthbuster/**/prompts.csv", recursive=True)
print("Encontrado:", origem)
shutil.copy(origem[0], f"{config.BRUTOS}/synthbuster/prompts.csv")
print(os.listdir(f"{config.BRUTOS}/synthbuster/"))

Encontrado: ['/content/synthbuster/synthbuster/prompts.csv']
['synthbuster.zip', 'inventario.csv', 'prompts.csv']


A parte do raise foi feita pelo PC


In [26]:
import pandas as pd
import os

raise_csv = pd.read_csv(f"{config.BRUTOS}/raise/RAISE_1k.csv")
prompts = pd.read_csv(f"{config.BRUTOS}/synthbuster/prompts.csv")

print("Colunas do prompts.csv:", prompts.columns.tolist())
print(prompts.head(3).to_string())

for g in ["dalle2", "midjourney-v5", "stable-diffusion-xl"]:
    print(g, sorted(os.listdir(f"/content/synthbuster/synthbuster/{g}"))[:3])

print("Colunas do RAISE:", raise_csv.columns.tolist())
print(raise_csv["File"].head(3).tolist())

Colunas do prompts.csv: ['image name (matching Raise-1k)', 'Prompt', 'Midjourney aspect ratio', 'Midjourney image choice', 'Glide/SD 1.x CFG scale', 'SD14 scale', 'Glide upsample temperature', 'Glide choice', 'SD 1.3 choice', 'SD 1.4 choice', 'sd2 resolution', 'sdxl width', 'sdxl height', 'sd2/sdxl steps', 'sd2/sdxl sampler', 'sd2/sdxl guider', 'sd2/sdxl guidance if vanilla scale', 'sdxl use refiner?', 'sdxl refiner steps', 'sdxl refiner sampler', 'sdxl refiner guider', 'sdxl refiner guidance if vanilla scale', 'sdxl finish denoising', 'dalle3 ar (not guaranteed)', 'Firefly styles', 'Firefly aspect ratio', 'Firefly guide strength', 'Firefly creativity', 'Firefly exif mode', 'Firefly manual exif aperture', 'Firefly manual exif shutter speed', 'Firefly manual exif FOV', 'Firefly generation URL']
  image name (matching Raise-1k)                                                                                                                                                                   